# Implementing Advantage-Actor Critic (A2C) - 2 pts

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel. 

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscal, take max between frames, skip frames, stack them together, prepares for PyTorch and normalizes to [0, 1]) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function.

In [ ]:
!pip install gymnasium==1.0.0
!pip install ale-py==0.10.2
!pip install opencv-python
!pip install gymnasium[other]

In [1]:
import numpy as np
from atari_wrappers import nature_dqn_env
import gymnasium as gym
from atari_wrappers import TensorboardSummaries

nenvs = 8    # change this if you have more than 8 CPU ;)
env = gym.vector.AsyncVectorEnv([lambda: nature_dqn_env("SpaceInvadersNoFrameskip-v4") for _ in range(nenvs)])
env = TensorboardSummaries(env, "spaceinvaders")


n_actions = env.single_action_space.n
obs, info = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32

A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]


Next, we will need to implement a model that predicts logits of policy distribution and critic value. Use shared backbone. You may use same architecture as in DQN task with one modification: instead of having a single output layer, it must have two output layers taking as input the output of the last hidden layer (one for actor, one for critic). 

Still it may be very helpful to make more changes:
* use orthogonal initialization with gain $\sqrt{2}$ and initialize biases with zeros;
* use more filters (e.g. 32-64-64 instead of 16-32-64);
* use two-layer heads for actor and critic or add a linear layer into backbone;

**Danger:** do not divide on 255, input is already normalized to [0, 1] in our wrappers!

In [2]:
import torch
import torch.nn as nn

class ConvBackbone(nn.Sequential):
    def __init__(self, c_in: int = 4) -> None:
        conv1 = nn.Conv2d(c_in, 32, (8, 8), 4)
        nn.init.orthogonal_(conv1.weight, 2**0.5)
        nn.init.zeros_(conv1.bias)

        conv2 = nn.Conv2d(32, 64, (4, 4), 2)
        nn.init.orthogonal_(conv2.weight, 2**0.5)
        nn.init.zeros_(conv2.bias)

        conv3 = nn.Conv2d(64, 64, (3, 3), 1)
        nn.init.orthogonal_(conv3.weight, 2**0.5)
        nn.init.zeros_(conv3.bias)

        super().__init__(
            conv1,
            nn.ReLU(),
            conv2,
            nn.ReLU(),
            conv3,
            nn.ReLU(),
            nn.Flatten(),
        )


class DuelingDqnHead(nn.Module):
    def __init__(self, n_actions, inp_size=64 * 7 * 7, hidden_size=512) -> None:
        super().__init__()
        linear1 = nn.Linear(inp_size, hidden_size)
        nn.init.orthogonal_(linear1.weight, 2**0.5)
        nn.init.zeros_(linear1.bias)

        linear2 = nn.Linear(hidden_size, n_actions)
        nn.init.orthogonal_(linear2.weight, 2**0.5)
        nn.init.zeros_(linear2.bias)

        self.adv_stream = nn.Sequential(
            linear1,
            nn.ReLU(),
            linear2
        )

        linear1 = nn.Linear(inp_size, hidden_size)
        nn.init.orthogonal_(linear1.weight, 2**0.5)
        nn.init.zeros_(linear1.bias)

        linear2 = nn.Linear(hidden_size, 1)
        nn.init.orthogonal_(linear2.weight, 2**0.5)
        nn.init.zeros_(linear2.bias)

        self.value_stream = nn.Sequential(
            linear1,
            nn.ReLU(),
            linear2
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.ndim == 2, x.shape  # (batch_size, n_features)
        value = self.value_stream(x)
        adv = self.adv_stream(x)
        return adv, value.squeeze(1)

class GradScalerFunctional(torch.autograd.Function):
    """
    A torch.autograd.Function works as Identity on forward pass
    and scales the gradient by scale_factor on backward pass.
    """
    @staticmethod
    def forward(ctx, input, scale_factor):
        ctx.scale_factor = scale_factor
        return input

    @staticmethod
    def backward(ctx, grad_output):
        scale_factor = ctx.scale_factor
        grad_input = grad_output * scale_factor
        return grad_input, None


class GradScaler(nn.Module):
    """
    An nn.Module incapsulating GradScalerFunctional
    """
    def __init__(self, scale_factor: float):
        super().__init__()
        self.scale_factor = scale_factor

    def forward(self, x):
        return GradScalerFunctional.apply(x, self.scale_factor)

class DQNetworkDueling(nn.Sequential):
    def __init__(self, c_in: int, n_actions: int) -> None:
        backbone = ConvBackbone(c_in=c_in)  # your code
        grad_scaler = GradScaler(1 / 2**0.5)  # Dueling DQN suggests do scale the gradient by 1 / sqrt(2)
        head = DuelingDqnHead(n_actions=n_actions)
        super().__init__(backbone, grad_scaler, head)

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a **dictionary** of all the arrays that are needed to interact with an environment and train the model.

**Important**: "actions" will be sent to environment, they must be numpy array or list, not PyTorch tensor.

Note: you can add more keys, e.g. it can be convenient to compute entropy right here.

In [3]:
from torch.distributions import Categorical

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
class Policy:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        '''
        input:
            inputs - numpy array, (batch_size x channels x width x height)
        output: dict containing keys ['actions', 'logits', 'log_probs', 'values']:
            'actions' - selected actions, numpy, (batch_size)
            'logits' - actions logits, tensor, (batch_size x num_actions)
            'log_probs' - log probs of selected actions, tensor, (batch_size)
            'values' - critic estimations, tensor, (batch_size)
        '''
        action_logs, values = self.model(torch.tensor(inputs).to(DEVICE))
        dist = Categorical(logits=action_logs)
        actions = dist.sample()
        log_probs =  dist.log_prob(actions)
        action_probs = torch.softmax(action_logs, dim=-1)
        entropy = -(action_probs * torch.log(action_probs + 1e-10)).sum(dim=-1)

        return {
            "actions": actions.detach().cpu().numpy(),
            "logits": action_logs,
            "log_probs": log_probs,
            "values": values,
            "entropy": entropy
        }

Next we will pass the environment and policy to a runner that collects rollouts from the environment. 
The class is already implemented for you.

In [4]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys 

* 'observations' 
* 'rewards' 
* 'dones'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment of specified length $T$ &mdash; the size of partial trajectory, or rollout length. Let's have a look at how it works.

In [5]:
model = DQNetworkDueling(4, n_actions).to(DEVICE)
policy = Policy(model)
runner = EnvRunner(env, policy, nsteps=5)

In [6]:
# generates new rollout
trajectory = runner.get_next()

In [7]:
# what is inside}
print(trajectory.keys())

dict_keys(['actions', 'logits', 'log_probs', 'values', 'entropy', 'observations', 'rewards', 'dones'])


In [8]:
# Sanity checks
assert 'logits' in trajectory, "Not found: policy didn't provide logits"
assert 'log_probs' in trajectory, "Not found: policy didn't provide log_probs of selected actions"
assert 'values' in trajectory, "Not found: policy didn't provide critic estimations"
assert trajectory['logits'][0].shape == (nenvs, n_actions), "logits wrong shape"
assert trajectory['log_probs'][0].shape == (nenvs,), "log_probs wrong shape"
assert trajectory['values'][0].shape == (nenvs,), "values wrong shape"

for key in trajectory.keys():
    assert len(trajectory[key]) == 5, \
    f"something went wrong: 5 steps should have been done, got trajectory of length {len(trajectory[key])} for '{key}'"

Now let's work with this trajectory a bit. To train the critic you will need to compute the value targets. It will also be used as an estimation of $Q$ for actor training.

You should use all available rewards for value targets, so the formula for the value targets is simple:

$$
\hat v(s_t) = \sum_{t'=0}^{T - 1}\gamma^{t'}r_{t+t'} + \gamma^T \hat{v}(s_{t+T}),
$$

where $s_{t + T}$ is the latest observation of the environment.

Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected. 
Thus, we can implement and use `ComputeValueTargets` callable. 

**Do not forget** to use `trajectory['dones']` flags to check if you need to add the value targets at the next step when 
computing value targets for the current step.

**Bonus (+0.5 pts):** implement [Generalized Advantage Estimation (GAE)](https://arxiv.org/pdf/1506.02438.pdf) instead; use $\lambda \approx 0.95$ or even closer to 1 in experiment. 

In [9]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy: Policy = policy
        self.gamma = gamma

    def __call__(self, trajectory, latest_observation):
        '''
        This method should modify trajectory inplace by adding 
        an item with key 'value_targets' to it
        
        input:
            trajectory - dict from runner
            latest_observation - last state, numpy, (num_envs x channels x width x height)
        '''
        _, latest_values = self.policy.model(torch.tensor(latest_observation).to(DEVICE))
        latest_values = latest_values.detach().cpu().numpy()
        value_targets = np.array(trajectory['rewards']) * self.gamma**np.arange(len(trajectory['rewards'])).reshape(-1, 1)
        value_targets = value_targets[::-1].cumsum(axis=0)[::-1]
        latest_values *= self.gamma**len(trajectory['rewards'])
        for i, v in enumerate(trajectory['dones']):
            value_targets[i] += ~v * latest_values
        trajectory['value_targets'] = value_targets

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `T * nenvs`.

You need to make sure that after this transformation `"log_probs"`, `"value_targets"`, `"values"` are 1-dimensional PyTorch tensors.

In [10]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory, latest_observation):
        # Modify trajectory inplace. 
        trajectory['log_probs'] = torch.concat(trajectory['log_probs'])
        trajectory['value_targets'] = torch.tensor(trajectory['value_targets'].reshape(-1)).to(DEVICE)
        trajectory['values'] = torch.concat(trajectory['values'])
        trajectory['entropy'] = torch.concat(trajectory['entropy'])

Let's do more sanity checks!

In [11]:
runner = EnvRunner(env, policy, nsteps=5, transforms=[ComputeValueTargets(policy),
                                                      MergeTimeBatch()])

trajectory = runner.get_next()

In [12]:
# More sanity checks
assert 'value_targets' in trajectory, "Value targets not found"
assert trajectory['log_probs'].shape == (5 * nenvs,)
assert trajectory['value_targets'].shape == (5 * nenvs,)
assert trajectory['values'].shape == (5 * nenvs,)

assert trajectory['log_probs'].requires_grad, "Gradients are not available for actor head!"
assert trajectory['values'].requires_grad, "Gradients are not available for critic head!"

Now is the time to implement the advantage actor critic algorithm itself. You can look into [Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and lectures ([part 1](https://www.youtube.com/watch?v=Ds1trXd6pos&list=PLkFD6_40KJIwhWJpGazJ9VSj9CFMkb79A&index=5), [part 2](https://www.youtube.com/watch?v=EKqxumCuAAY&list=PLkFD6_40KJIwhWJpGazJ9VSj9CFMkb79A&index=6)) by Sergey Levine.

In [13]:
from collections import defaultdict
from torch.nn.utils import clip_grad_norm_

class A2C:
    def __init__(self, policy, optimizer, value_loss_coef=0.25, entropy_coef=0.01, max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
    
    def loss(self, trajectory, write):
        # compute all losses
        # do not forget to use weights for critic loss and entropy loss
        critic_loss = (trajectory['values'] - trajectory['value_targets']).pow(2).mean()
        advantages = trajectory['value_targets'] - trajectory['values']
        policy_loss = -(trajectory['log_probs'] * advantages.detach()).sum() / len(trajectory['rewards'][0])
        entropy_loss = trajectory['entropy'].mean()

        # log all losses
        write('losses', {
            'policy loss': policy_loss.item(),
            'critic loss': critic_loss.item(),
            'entropy loss': entropy_loss.item()
        })
        
        # additional logs
        write('critic/advantage', advantages.mean().item())
        write('critic/values', {
            'value predictions': trajectory['values'].mean().item(),
            'value targets':     trajectory['value_targets'].mean().item(),
        })
        
        loss = policy_loss + self.value_loss_coef * critic_loss + self.entropy_coef * entropy_loss
        # return scalar loss
        return loss        

    def train(self, runner):
        # collect trajectory using runner
        # compute loss and perform one step of gradient optimization
        # do not forget to clip gradients
        trajectory = runner.get_next()
        loss = self.loss(trajectory, runner.write)

        self.optimizer.zero_grad()
        loss.backward()

        grad_norm = clip_grad_norm_(self.policy.model.parameters(), self.max_grad_norm)
        self.optimizer.step()
        # use runner.write to log scalar to tensorboard
        runner.write('gradient norm', grad_norm)

Now you can train your model. For optimization we suggest you use RMSProp with learning rate 7e-4 (you can also linearly decay it to 0), smoothing constant (alpha in PyTorch) equal to 0.99 and epsilon equal to 1e-5.

We recommend to train for at least 10 million environment steps across all batched environments (takes ~3 hours on a single GTX1080 with 8 CPU). It should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last episodes in each environment in the batch) of about 600. **Your goal is to reach 500**.

Notes:
* if your reward is stuck at ~200 for more than 2M steps then probably there is a bug
* if your gradient norm is >10 something probably went wrong
* make sure your `entropy loss` is negative, your `critic loss` is positive
* make sure you didn't forget `.detach` in losses where it's needed
* `actor loss` should oscillate around zero or near it; do not expect loss to decrease in RL ;)
* you can experiment with `nsteps` ("rollout length"); standard rollout length is 5 or 10. Note that this parameter influences how many algorithm iterations is required to train on 10M steps (or 40M frames --- we used frameskip in preprocessing).

In [14]:
model = DQNetworkDueling(4, n_actions).to('cuda')
policy = Policy(model)
runner = EnvRunner(env, policy, nsteps=10, transforms=[ComputeValueTargets(policy),
                                                      MergeTimeBatch()])

optimizer = torch.optim.RMSprop(policy.model.parameters(), lr=7e-4, alpha=0.99, eps=1e-5)

a2c = A2C(policy, optimizer)

In [ ]:
for _ in range(100000):
    a2c.train(runner)

In [ ]:
# save your model just in case 
torch.save(model.state_dict(), "A2C")    

In [ ]:
env.close()

## Evaluation

In [ ]:
env = nature_dqn_env("SpaceInvadersNoFrameskip-v4", clip_reward=False, episodic_life=False)

In [ ]:
def evaluate(env, policy, n_games=1, t_max=10000):
    '''
    Plays n_games and returns rewards
    '''
    rewards = []
    
    for _ in range(n_games):
        s, info = env.reset()
        
        R = 0
        for _ in range(t_max):
            action = policy.act(np.array([s]))["actions"][0]
            
            s, r, term, trank, _ = env.step(action)
            
            R += r
            if term or trank:
                break

        rewards.append(R)
    return np.array(rewards)

In [ ]:
# evaluation will take some time!
sessions = evaluate(env, policy, n_games=30)
score = sessions.mean()
print(f"Your score: {score}")

assert score >= 500, "Needs more training?"
print("Well done!")

In [ ]:
env.close()

## Record

In [ ]:
env_monitor = nature_dqn_env("SpaceInvadersNoFrameskip-v4", monitor=True, clip_reward=False, episodic_life=False)

In [ ]:
# record sessions
sessions = evaluate(env_monitor, policy, n_games=3)

In [ ]:
# rewards for recorded games
sessions

In [ ]:
env_monitor.close()